# Brique 0 — Structurer un projet Python comme un pro

Tu as sûrement ouvert des repos GitLab chez Cozycozy et vu des dossiers `src/`, `lib/`, des fichiers
`requirements.txt`, `.env`, `.gitignore`, parfois `pyproject.toml`. Ce notebook explique **pourquoi** chacun de
ces éléments existe, puis tu les mets en pratique en structurant un petit projet de zéro.

Rappel implicite de `00_cours.ipynb` : Python n'a pas de compilateur qui vérifie les types et les dépendances
avant l'exécution. Toutes les conventions de ce notebook existent pour **compenser** ça : rendre explicite ce
que le langage ne force pas lui-même (quelles dépendances, quelle version, quels secrets, quels fichiers
suivre).

## 1. Pourquoi un environnement virtuel (`venv`) avant tout le reste

Si tu installes toutes tes bibliothèques directement sur ton système (`pip install pandas`), elles sont
partagées par **tous** tes projets Python. Le jour où le projet A a besoin de `pandas==1.5` et le projet B de
`pandas==2.1`, ils se marchent dessus — impossible d'avoir les deux en même temps globalement.

Un **environnement virtuel** est un dossier qui contient sa propre copie de l'interpréteur Python et ses propres
bibliothèques, isolée du reste du système. Chaque projet a le sien.

```bash
python -m venv .venv          # crée l'environnement dans le dossier .venv
# Windows :
.venv\Scripts\activate
# macOS / Linux :
source .venv/bin/activate
```

Une fois activé, `pip install ...` installe **dans ce dossier**, pas sur le système. C'est pour ça que `.venv/`
se retrouve toujours dans `.gitignore` (section 5) : ce n'est pas du code, c'est une copie locale et volumineuse
de dépendances, reconstructible à tout moment.

## 2. `requirements.txt` — déclarer ce dont le projet dépend

Puisque les dépendances ne sont plus installées globalement, il faut un fichier qui dise explicitement "ce
projet a besoin de ces bibliothèques, à ces versions". C'est `requirements.txt` : une ligne par dépendance.

```
pandas==2.1.4
numpy>=1.26
scikit-learn
```

- `==` fige une version exacte (reproductibilité totale, utile en production).
- `>=` fixe un minimum (plus souple, risque d'incompatibilité future).
- Aucune contrainte = toujours la dernière version (pratique en apprentissage, dangereux en production :
  le projet peut casser du jour au lendemain sans qu'on ait rien changé).

N'importe qui (ou un serveur de déploiement) peut alors reconstruire exactement le même environnement :

```bash
pip install -r requirements.txt
```

**Pourquoi tu vois parfois `pyproject.toml` à la place** : c'est l'évolution moderne, un seul fichier qui
déclare à la fois les dépendances, les métadonnées du projet (nom, version, auteur) et la configuration des
outils (formatteur, linter). Des outils comme `poetry` ou `uv` (que tu as croisé dans `devenir_ml_engineer.md`
via les labs MDC) s'appuient sur ce fichier. Pour ce parcours, `requirements.txt` suffit ; retiens juste que
`pyproject.toml` répond au même besoin, en plus complet.

## 3. `src/` — pourquoi le code n'est pas directement à la racine

Tu verras deux organisations possibles :

```
projet_a/              projet_b/
  mon_package/            src/
    __init__.py             mon_package/
    module.py                  __init__.py
  tests/                      module.py
  requirements.txt         tests/
                          requirements.txt
```

Le layout **`src/`** (le deuxième) est aujourd'hui la convention recommandée. Raison concrète : sans `src/`, le
dossier racine du projet est lui-même importable par erreur (parce qu'il est dans le `PYTHONPATH` quand tu lances
un script depuis là), ce qui peut masquer un vrai bug d'installation du package — tes tests passent en local
mais échoueraient si quelqu'un installait vraiment ton package ailleurs. Avec `src/`, ton code n'est accessible
qu'en l'important **comme un vrai package installé**, donc si ça marche en local, ça marche partout.

C'est l'équivalent en esprit du dossier `src/` que tu connais déjà en Java (tes propres TD en 1A/2A ont cette
structure `src/` + `bin/` pour le compilé) : séparer le code source du reste (configuration, données, sorties).

## 4. `lib/` — rare aujourd'hui, mais tu le verras parfois

Tu as toi-même un dossier `lib/` dans `Java example 1A` (`TC.jar`, `MacLib.jar`) : en Java, ce sont des
bibliothèques externes fournies en `.jar`, à côté du code, parce que le cours ne t'a pas appris un gestionnaire
de dépendances comme Maven.

En Python, un dossier `lib/` à la racine d'un repo GitLab a généralement l'un de ces rôles :

- du **code interne partagé** entre plusieurs scripts ou services du même repo (pas une vraie bibliothèque
  externe, juste une façon de séparer "code réutilisable" de "points d'entrée") ;
- plus rarement, des dépendances **vendorisées** (copiées directement dans le repo plutôt qu'installées via pip),
  pour des cas particuliers (pas de connexion internet au déploiement, dépendance modifiée à la main).

Avec `pip` et `requirements.txt`, le vendoring est devenu rare : la convention moderne pour "du code interne
partagé" est plutôt un sous-dossier dans `src/` qu'un `lib/` séparé. Si tu vois `lib/` dans un repo de ta boîte,
regarde ce qu'il contient avant de supposer — c'est souvent historique.

## 5. `.gitignore` — dire à Git ce qu'il ne doit jamais suivre

Trois catégories de fichiers n'ont rien à faire dans Git :

1. **Reconstructible automatiquement** : `__pycache__/`, `*.pyc` (vu en `00_cours.ipynb`), `.venv/` — les
   regénérer ne coûte rien, les versionner gonflerait le repo pour rien.
2. **Spécifique à ta machine** : fichiers de ton éditeur (`.vscode/` selon les équipes), fichiers temporaires.
3. **Secret** : `.env` (section suivante) — s'il est commité une seule fois, le secret est dans l'historique
   Git pour toujours, même si tu le supprimes après (il faudrait réécrire l'historique, ou pire, révoquer et
   régénérer le secret).

```
__pycache__/
*.pyc
.venv/
.env
.ipynb_checkpoints/
```

Regarde le `.gitignore` de ce dépôt (`cs-mastery-path/.gitignore`), déjà en place.

## 6. `.env` — séparer la configuration du code

Un mot de passe de base de données, une clé d'API, une URL différente entre dev et prod : ces valeurs ne doivent
**jamais** être écrites en dur dans le code (sinon elles finissent dans Git, donc potentiellement publiques, et
changer d'environnement veut dire modifier le code).

La convention : un fichier `.env` à la racine, ignoré par Git, contenant des paires clé=valeur :

```
DATABASE_URL=postgresql://localhost:5432/madb
API_KEY=sk-abc123...
```

Le code le lit via une bibliothèque comme `python-dotenv`, sans jamais écrire la vraie valeur :

```python
from dotenv import load_dotenv
import os

load_dotenv()                       # lit le fichier .env et remplit les variables d'environnement
api_key = os.environ["API_KEY"]     # jamais la valeur en dur dans le fichier .py
```

Le repo contient souvent un `.env.example` (celui-là **est** commité), qui liste les clés attendues avec des
valeurs factices, pour que quelqu'un qui clone le repo sache quoi remplir.

## 7. Récapitulatif — à quoi ressemble un repo bien structuré

```
mon_projet/
├── src/
│   └── mon_projet/
│       ├── __init__.py
│       └── core.py
├── tests/
│   └── test_core.py
├── .env.example
├── .gitignore
├── requirements.txt
└── README.md
```

Ce `cs-mastery-path` que tu es en train de parcourir suit volontairement une version simplifiée de cette logique
(`requirements.txt`, `.gitignore`, `README.md` à la racine) — simplifiée parce qu'ici le "code" ce sont des
notebooks pédagogiques, pas un package à installer.

## 8. À toi — exercice guidé

Pas dans ce notebook, mais sur ta machine (ou dans Colab avec le terminal intégré) : crée un dossier
`mon_premier_repo/` et structure-le en suivant les étapes ci-dessous. Coche chaque étape une fois faite.

1. [ ] Crée le dossier et initialise un dépôt Git (`git init`).
2. [ ] Crée un environnement virtuel `.venv` et active-le.
3. [ ] Crée `src/mon_premier_repo/__init__.py` (fichier vide) et `src/mon_premier_repo/core.py` avec une
   fonction `bonjour(nom: str) -> str` qui retourne `f"Bonjour {nom}"`.
4. [ ] Crée `requirements.txt` (même vide pour l'instant, juste le fichier).
5. [ ] Crée `.gitignore` avec au moins `__pycache__/`, `.venv/`, `.env`.
6. [ ] Crée un `.env` avec une variable `NOM_PAR_DEFAUT=Youssef`, et un `.env.example` avec
   `NOM_PAR_DEFAUT=ton_nom_ici`.
7. [ ] Vérifie avec `git status` que `.venv/` et `.env` **n'apparaissent pas** comme fichiers à committer — si
   l'un des deux apparaît, ton `.gitignore` a un problème, corrige-le avant de continuer.
8. [ ] Fais un premier commit propre (`git add`, `git commit`) et vérifie avec `git log` qu'il est bien là.

Une fois que tu as fait ça une fois à la main, tu comprends ce que fait réellement un générateur de projet
(`cookiecutter`, `poetry new`, etc.) quand tu l'utiliseras plus tard : il automatise exactement ces étapes.

---

**Brique 0 terminée.** Coche-la dans [ROADMAP.md](../ROADMAP.md), puis on passe à la Brique 1 (rappel Python de
base) quand tu es prêt.